# 01 — Data Audit

In [1]:
# Cell 1 — setup


import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent / "src"))
from scorelab import config, data
con = data.connect()            # read-only DuckDB connection to data/lc.duckdb

ModuleNotFoundError: No module named 'duckdb'

In [ ]:
# Cell 2 — sanity check
print(con.sql("SHOW TABLES"))
print(con.sql("SELECT COUNT(*) AS n, MIN(issue_d), MAX(issue_d) FROM accepted"))

In [ ]:
# Cell 3 — common imports and settings for this notebook
import json, warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.width", 180); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 200)
from scorelab import target, attributes as A, binning as B, scorecard as S, validation as V, fairness as F
from scorelab import challengers as C, reject_inference as R, monitoring as M, report as REP
MODELS = config.MODELS_DIR; MODELS.mkdir(exist_ok=True)
SAMPLE_FRAC = None     # None = full data (recommended for the fitting notebooks); e.g. 0.10 for a quick pass
cols = [r[0] for r in con.execute("DESCRIBE v_target").fetchall()]
assert "label" in cols, "v_target is the old stub: close this kernel and run `python scripts/refresh_views.py`"
def load_frame(splits=("train", "validate", "oot"), **kw):
    """Observable, non-excluded loans with attributes built; indeterminates dropped unless asked."""
    f = target.modeling_frame(con, splits=splits, sample_frac=SAMPLE_FRAC, **kw)
    return A.build_attributes(f)
MEANINGS = {a.name: a.meaning for a in A.REGISTRY}
DIRECTIONS = {a.name: a.direction for a in A.REGISTRY}

## What this notebook settles

Two decisions and a set of findings for the model document. **Decision A — scope:** which issue years, terms and application types the project models (the split in `config.py` assumes 2012–2016 for fitting, 2017Q1–2018Q4 for monitoring). **Decision B — fields with structural gaps:** bureau fields that only exist for late vintages or are mostly missing in the training years are not attributes for a card that must be stable across years; list them here and NB03 drops them.

Write your reasoning in the decision cells before running the evidence below them.

In [ ]:
# Evidence — what is in the database
display(data.row_counts(con))
display(con.sql("""
    SELECT year(issue_d) AS vintage_year, TRY_CAST(regexp_extract(term, '[0-9]+') AS INTEGER) AS term_months,
           COUNT(*) AS n, ROUND(AVG(loan_amnt), 0) AS mean_amount
    FROM accepted WHERE year(issue_d) >= 2012 GROUP BY 1, 2 ORDER BY 1, 2
""").df())
display(con.sql("SELECT loan_status, COUNT(*) AS n FROM accepted GROUP BY 1 ORDER BY 2 DESC").df())

In [2]:
# Evidence — field availability by vintage for every raw input the attribute registry reads
raw_inputs = sorted({c for a in A.REGISTRY for c in a.sources if c not in ("issue_d",)})
avail = data.missingness_by_vintage(raw_inputs, con).drop(index="n")   # share MISSING per field and year
display(avail.style.background_gradient(cmap="Reds", axis=None) if hasattr(avail, "style") else avail)
train_years = list(config.TRAIN_VINTAGES)
structural = avail[(avail.loc[:, train_years] > 0.95).any(axis=1)]            # absent for a whole training year
reporting_start = avail[(avail[train_years[0]] > 0.30) & (avail[train_years[-1]] < 0.05)]   # began mid-way through the first training year
print("Absent for an entire training year (structural):"); display(structural)
print("Reporting started during the first training year:"); display(reporting_start)
avail.to_csv(config.DOCS_DIR / "audit" / "field_missingness_by_vintage.csv")

NameError: name 'A' is not defined

In [3]:
# Evidence — when did the reporting-start fields begin? (by issue month of the first training year)
if len(reporting_start):
    probe = reporting_start.index[:6].tolist()
    exprs = ", ".join(f"ROUND(AVG(CASE WHEN {c} IS NULL THEN 1 ELSE 0 END), 3) AS {c}" for c in probe)
    display(con.sql(f"SELECT issue_d, COUNT(*) AS n, {exprs} FROM v_accepted_asof WHERE year(issue_d) = {train_years[0]} GROUP BY 1 ORDER BY 1").df())

NameError: name 'reporting_start' is not defined

In [4]:
# Evidence — ranges and tails of the main numeric fields (caps and reporting artefacts)
key = ["revol_util", "bc_util", "all_util", "percent_bc_gt_75", "total_acc", "open_acc", "inq_last_6mths", "delinq_2yrs",
       "mths_since_last_delinq", "revol_bal", "tot_cur_bal", "avg_cur_bal", "bc_open_to_buy", "mo_sin_old_rev_tl_op"]
display(data.numeric_profile(key, con, years=tuple(config.TRAIN_VINTAGES) + (config.VALIDATE_VINTAGE, config.OOT_VINTAGE)))

NameError: name 'data' is not defined

In [5]:
# Evidence — categorical context fields and the incumbent score
for c in ("purpose", "application_type", "addr_state"):
    print(c); display(data.categorical_profile(c, con, top=12))
display(data.incumbent_coverage(con))

purpose


NameError: name 'data' is not defined

In [ ]:
# Evidence — the rejected-applications file (through-the-door context for NB06)
display(con.sql("""
    SELECT year(application_date) AS year, COUNT(*) AS n_rejected,
           ROUND(AVG(CASE WHEN risk_score IS NULL OR risk_score = 0 THEN 1 ELSE 0 END), 4) AS risk_score_missing,
           ROUND(AVG(CASE WHEN risk_score > 0 THEN risk_score END), 1) AS mean_risk_score
    FROM rejected GROUP BY 1 ORDER BY 1
""").df())
display(R.through_the_door_profile(con, years=(2014, 2015, 2016)))

### Decision A — Scope

Years, terms, application types. A block of bureau fields is unreported for the first part of 2012 and complete from the month it started: either begin the training window at that month (or at 2013, editing `config.TRAIN_VINTAGES` and re-running `scripts/refresh_views.py`), or drop those fields. Is a 60-month loan the same product as a 36-month loan for a 24-month score, or a segment? Are joint applications (a few thousand in 2016 only) in or out?

**Your reasoning:**

_(write here)_

### Decision B — Fields with structural gaps

From the missingness table: fields absent for an entire training year (the 2016-only block: `inq_last_12m`, `open_rv_24m`, `il_util`, `all_util`, `max_bal_bc`, `mths_since_rcnt_il`, `open_il_24m`) cannot be attributes of a card trained on 2012–2014. Fields that are missing because the consumer has no such event (`mths_since_last_delinq` = never delinquent) are *not* gaps: the missing bin is information. Decide the reporting-start block under Decision A. List the structural fields for NB03.

**Your reasoning and the list:**

_(write here)_

In [ ]:
# Record Decision B here so NB03 can read it (edit the list, then run)
STRUCTURAL_GAP_FIELDS = list(structural.index)      # default: absent for a whole training year; add the reporting-start block here if you keep 2012
(MODELS / "audit_decisions.json").write_text(json.dumps({"structural_gap_fields": STRUCTURAL_GAP_FIELDS}, indent=1), encoding="utf-8")
print("saved", MODELS / "audit_decisions.json", "->", STRUCTURAL_GAP_FIELDS)